In [0]:
%sql
-- Remove the '$' sign from the amount column and rewrite the table in place
CREATE OR REPLACE TABLE payments.default.pay_transactions AS
SELECT 
  * EXCEPT (amount),
  CAST(REPLACE(amount, '$', '') AS DECIMAL(10,2)) AS amount
FROM payments.default.pay_transactions;

In [0]:
%sql
CREATE OR REPLACE TABLE payments.default.pay_transactions AS
SELECT *, CASE WHEN merchant_city = 'ONLINE' THEN 'ONLINE'
            ELSE 'OFFLINE'
        END AS is_online
  FROM payments.default.pay_transactions;

In [0]:
%sql
-- =====================================================================
-- Feature engineering: one CREATE OR REPLACE per table, no temp views.
-- Each output table is built in a single statement using CTEs.
--   payments.default.pay_users_features
--   payments.default.pay_transactions_features
-- =====================================================================


-- =====================================================================
-- TABLE 1: USERS
-- =====================================================================
CREATE OR REPLACE TABLE payments.default.pay_users_features AS
WITH
users_clean AS (
  -- "$12,345" -> 12345.0 (CAST to STRING so it also works if already numeric)
  SELECT
    id, address, gender, birth_month, birth_year, current_age, retirement_age,
    latitude, longitude, credit_score, num_credit_cards,
    TRY_CAST(REGEXP_REPLACE(CAST(yearly_income     AS STRING), '[$,]', '') AS DOUBLE) AS yearly_income,
    TRY_CAST(REGEXP_REPLACE(CAST(total_debt        AS STRING), '[$,]', '') AS DOUBLE) AS total_debt,
    TRY_CAST(REGEXP_REPLACE(CAST(per_capita_income AS STRING), '[$,]', '') AS DOUBLE) AS per_capita_income
  FROM payments.default.pay_users
),
users_ratios AS (
  SELECT
    *,
    ROUND(total_debt    / NULLIF(yearly_income, 0), 2)     AS debt_to_income,
    ROUND(yearly_income / NULLIF(per_capita_income, 0), 2) AS income_vs_area,
    GREATEST(retirement_age - current_age, 0)              AS years_to_retirement,
    current_age >= retirement_age                          AS is_retired,
    ROUND(total_debt / NULLIF(num_credit_cards, 0), 0)     AS debt_per_card
  FROM users_clean
)
SELECT
  *,
  CASE
    WHEN credit_score IS NULL THEN 'Unknown'
    WHEN credit_score < 580   THEN 'Poor'
    WHEN credit_score < 670   THEN 'Fair'
    WHEN credit_score < 740   THEN 'Good'
    WHEN credit_score < 800   THEN 'Very Good'
    ELSE 'Excellent'
  END AS credit_band,

  CASE
    WHEN birth_year IS NULL THEN 'Unknown'
    WHEN birth_year >= 1997 THEN 'Gen Z'
    WHEN birth_year >= 1981 THEN 'Millennial'
    WHEN birth_year >= 1965 THEN 'Gen X'
    WHEN birth_year >= 1946 THEN 'Boomer'
    ELSE 'Silent'
  END AS generation,

  -- ASSUMPTION: 'Medio' rule (0.8) reconstructed from a truncated cell
  CASE
    WHEN credit_score IS NULL AND debt_to_income IS NULL THEN 'Non valutabile'
    WHEN credit_score < 620 OR debt_to_income > 1.5      THEN 'Alto'
    WHEN credit_score < 700 OR debt_to_income > 0.8      THEN 'Medio'
    ELSE 'Basso'
  END AS credit_risk_level,

  CASE
    WHEN yearly_income IS NULL OR total_debt IS NULL THEN 'Non valutabile'
    WHEN total_debt > yearly_income * 3
      OR (num_credit_cards >= 6 AND yearly_income < per_capita_income) THEN 'Critical'
    WHEN total_debt > yearly_income * 2
      OR yearly_income > per_capita_income * 5                         THEN 'Suspicious'
    ELSE 'Normal'
  END AS fraud_risk_level
FROM users_ratios;


-- =====================================================================
-- TABLE 2: TRANSACTIONS
-- =====================================================================
CREATE OR REPLACE TABLE payments.default.pay_transactions_features AS
WITH
tx_clean AS (
  -- "$-77.00" -> -77.0
  SELECT
    * EXCEPT (amount),
    TRY_CAST(REGEXP_REPLACE(CAST(amount AS STRING), '[$,]', '') AS DOUBLE) AS amount
  FROM payments.default.pay_transactions
),
city_totals AS (
  SELECT
    merchant_city,
    merchant_state,
    SUM(amount) AS city_total_amount
  FROM tx_clean
  WHERE merchant_city IS NOT NULL
    AND TRIM(merchant_city) <> ''
    AND UPPER(TRIM(merchant_city)) <> 'ONLINE'
    AND amount IS NOT NULL
  GROUP BY merchant_city, merchant_state
),
city_groups AS (
  SELECT
    merchant_city,
    merchant_state,
    CASE
      WHEN CUME_DIST() OVER (ORDER BY city_total_amount) <= 1/3 THEN 'Low Level'
      WHEN CUME_DIST() OVER (ORDER BY city_total_amount) <= 2/3 THEN 'Middle Level'
      ELSE 'High Level'
    END AS city_amount_level
  FROM city_totals
),
customer_aggregation AS (
  SELECT
    client_id,
    COUNT(*)    AS customer_n_transactions,
    SUM(amount) AS customer_total_amount
  FROM tx_clean
  WHERE client_id IS NOT NULL
    AND amount IS NOT NULL
  GROUP BY client_id
),
customer_segments AS (
  SELECT
    client_id,
    customer_n_transactions,
    customer_total_amount,
    CASE
      WHEN CUME_DIST() OVER (ORDER BY customer_n_transactions) <= 1/3 THEN 'Bassa frequenza'
      WHEN CUME_DIST() OVER (ORDER BY customer_n_transactions) <= 2/3 THEN 'Media frequenza'
      ELSE 'Alta frequenza'
    END AS customer_frequency_level,
    -- ASSUMPTION: labels reconstructed from a truncated cell
    CASE
      WHEN CUME_DIST() OVER (ORDER BY customer_total_amount) <= 1/3 THEN 'Bassa spesa'
      WHEN CUME_DIST() OVER (ORDER BY customer_total_amount) <= 2/3 THEN 'Media spesa'
      ELSE 'Alta spesa'
    END AS customer_monetary_level
  FROM customer_aggregation
)
SELECT
  t.*,
  CASE
    WHEN UPPER(TRIM(t.merchant_city)) = 'ONLINE' THEN 'Online'
    ELSE COALESCE(c.city_amount_level, 'Città non disponibile')
  END AS city_amount_level,
  s.customer_n_transactions,
  s.customer_total_amount,
  COALESCE(s.customer_frequency_level, 'Cliente non disponibile') AS customer_frequency_level,
  COALESCE(s.customer_monetary_level,  'Cliente non disponibile') AS customer_monetary_level
FROM tx_clean AS t
LEFT JOIN city_groups AS c
  ON  t.merchant_city = c.merchant_city
  AND t.merchant_state <=> c.merchant_state
LEFT JOIN customer_segments AS s
  ON t.client_id = s.client_id;